### 0 단계: 기본 설정 및 준비

In [ ]:
# ============================================================
# [CELL 0] 패키지 설치
# - 이 노트북은 start_RAG.ipynb의 기본 흐름을 확장한 버전입니다.
# - OpenAI + LangChain + Chroma + RAGAS 기반으로 구성합니다.
# - retriever 관련 일부 모듈은 최근 버전에서 langchain-classic으로 분리되어
#   있으므로 함께 설치합니다.
# - reranker / hybrid retrieval / ragas 평가까지 한 번에 사용할 수 있도록
#   필요한 패키지를 미리 설치합니다.
# ============================================================

# !pip install -U langchain langchain-core langchain-community langchain-openai
# !pip install -U langchain-text-splitters langchain-experimental langchain-classic
# !pip install -U chromadb pypdf python-dotenv tiktoken
# !pip install -U rank-bm25 sentence-transformers datasets ragas
# !pip install -U langchain-experimental
# !pip install sentence-transformers
# !pip install ragas==0.1.21 --force-reinstall

In [1]:
# ============================================================
# [CELL 1] 환경변수 로드 및 공통 설정
# - start_RAG.ipynb에서 하던 dotenv 로딩 방식을 그대로 사용합니다.
# - Gemini는 제외하고 OpenAI만 사용합니다.
# - 아래 DOCUMENT_PATH만 바꾸면 PDF / TXT / CSV 중심으로 실험할 수 있게 구성합니다.
# - COLLECTION_PREFIX는 Chroma 컬렉션 이름 충돌을 피하기 위해 사용합니다.
# ============================================================

from dotenv import load_dotenv
import os
import uuid

# 본인 환경에 맞게 수정
load_dotenv(r"D:\PyProject\env_keys\.env")

openai_key = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai_key

# -----------------------------
# 실험 대상 문서 경로
# 예시:
# DOCUMENT_PATH = "./data/Demian.pdf"
# DOCUMENT_PATH = "./data/state_of_the_union.txt"
# DOCUMENT_PATH = "./data/titanic.csv"
# -----------------------------
DOCUMENT_PATH = "./data/Demian.pdf"

# Chroma collection 이름 충돌 방지용 prefix
COLLECTION_PREFIX = "start_rag_upgrade"

# 기본 chunk 설정
DEFAULT_CHUNK_SIZE = 500
DEFAULT_CHUNK_OVERLAP = 50

# 검색 기본값
TOP_K = 4
FETCH_K = 12

print("OPENAI_API_KEY loaded:", bool(openai_key))
print("DOCUMENT_PATH:", DOCUMENT_PATH)

OPENAI_API_KEY loaded: True
DOCUMENT_PATH: ./data/Demian.pdf


In [3]:
# ============================================================
# [CELL 2] 공통 import 및 LLM / Embedding 준비
# - start_RAG의 핵심 구성요소(Loader -> Splitter -> Embedding -> VectorStore -> Retriever)를
#   유지하면서, 이후 단계별 기법을 쉽게 꽂을 수 있도록 공통 모듈을 한 번에 준비합니다.
# - answer 생성은 ChatOpenAI(gpt-4o-mini), 임베딩은 text-embedding-3-small 기준입니다.
# ============================================================

import re
import json
import math
import textwrap
from pathlib import Path
from typing import List, Dict, Any, Callable

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    CSVLoader,
)

from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Semantic chunking
from langchain_experimental.text_splitter import SemanticChunker

# Advanced retrievers
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever

# BM25:Best Matching 알고리즘 25번 버전
# 정보 검색(IR)에서 가장 유명한 랭킹 알고리즘 중 하나 Best Matching, BM25는 단순 TF-IDF보다 발전된 형태로
from langchain_community.retrievers import BM25Retriever

# Reranker
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Evaluation
import pandas as pd
from datasets import Dataset

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key=openai_key
)

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=openai_key
)

print("LLM / Embedding 준비 완료")

LLM / Embedding 준비 완료


In [4]:
# ============================================================
# [CELL 3] 공통 유틸리티 함수
# - 이 셀은 이후 모든 단계에서 공통으로 재사용할 helper 모음입니다.
# - 주요 역할:
#   1) 문서 로딩
#   2) 메타데이터 강화
#   3) vectorstore 생성
#   4) 문서들을 context 문자열로 묶기
#   5) retriever 결과를 바탕으로 최종 답변 생성
# - 즉, 이후 단계는 "retriever만 바꿔 끼우는" 방식으로 비교하기 쉽게 설계합니다.
# ============================================================

def detect_file_type(path: str) -> str:
    ext = Path(path).suffix.lower()
    return ext.replace(".", "")

def load_documents(path: str) -> List[Document]:
    ext = Path(path).suffix.lower()

    if ext == ".pdf":
        loader = PyPDFLoader(path)
        docs = loader.load()
    elif ext in [".txt", ".md"]:
        loader = TextLoader(path, encoding="utf-8")
        docs = loader.load()
    elif ext == ".csv":
        loader = CSVLoader(path, encoding="utf-8")
        docs = loader.load()
    else:
        raise ValueError(f"지원하지 않는 파일 형식입니다: {ext}")

    return docs

def simple_keywords(text: str, top_n: int = 8) -> List[str]:
    tokens = re.findall(r"[가-힣A-Za-z0-9]{2,}", text.lower())
    stop = {
        "그리고", "그러나", "하지만", "입니다", "있는", "하는", "대한", "에서",
        "with", "that", "this", "from", "have", "were", "what", "when", "where",
        "which", "shall", "into", "also", "their", "there", "about"
    }
    freq = {}
    for t in tokens:
        if t not in stop:
            freq[t] = freq.get(t, 0) + 1
    return [k for k, _ in sorted(freq.items(), key=lambda x: x[1], reverse=True)[:top_n]]

def guess_section_title(text: str) -> str:
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return "unknown"
    first = lines[0]
    return first[:80]

def enrich_metadata(
    docs: List[Document],
    source_path: str
) -> List[Document]:
    file_type = detect_file_type(source_path)
    file_name = Path(source_path).name

    enriched = []
    for i, doc in enumerate(docs):
        text = doc.page_content or ""
        meta = dict(doc.metadata) if doc.metadata else {}

        meta["source_path"] = source_path
        meta["source_file"] = file_name
        meta["file_type"] = file_type
        meta["doc_id"] = i
        meta["section_title"] = guess_section_title(text)
        meta["keywords"] = simple_keywords(text, top_n=8)

        # PDF라면 page 메타데이터가 들어있는 경우가 많으므로 보정
        if "page" in meta:
            meta["page_num"] = meta["page"]

        enriched.append(Document(page_content=text, metadata=meta))

    return enriched

def make_recursive_chunks(
    docs: List[Document],
    chunk_size: int = DEFAULT_CHUNK_SIZE,
    chunk_overlap: int = DEFAULT_CHUNK_OVERLAP
) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    split_docs = splitter.split_documents(docs)

    for idx, d in enumerate(split_docs):
        d.metadata["chunk_id"] = idx
        d.metadata["chunking"] = "recursive"

    return split_docs

def make_semantic_chunks(docs: List[Document]) -> List[Document]:
    # SemanticChunker는 문맥 전환 지점을 기준으로 잘라서
    # 단순 고정 길이 split보다 의미 단위가 잘 보존되도록 합니다.
    all_text = "\n\n".join([d.page_content for d in docs if d.page_content.strip()])

    splitter = SemanticChunker(
        embeddings=embedding_model,
        breakpoint_threshold_type="percentile"
    )
    split_docs = splitter.create_documents([all_text])

    for idx, d in enumerate(split_docs):
        d.metadata["chunk_id"] = idx
        d.metadata["chunking"] = "semantic"
        d.metadata["source_path"] = docs[0].metadata.get("source_path", DOCUMENT_PATH)
        d.metadata["source_file"] = docs[0].metadata.get("source_file", Path(DOCUMENT_PATH).name)
        d.metadata["file_type"] = docs[0].metadata.get("file_type", detect_file_type(DOCUMENT_PATH))

    return split_docs

def build_chroma(
    docs: List[Document],
    collection_name: str
):
    return Chroma.from_documents(
        documents=docs,
        embedding=embedding_model,
        collection_name=f"{COLLECTION_PREFIX}_{collection_name}_{uuid.uuid4().hex[:8]}"
    )

def docs_to_context(docs: List[Document]) -> str:
    blocks = []
    for i, d in enumerate(docs, start=1):
        meta = d.metadata or {}
        head = {
            "idx": i,
            "source_file": meta.get("source_file"),
            "page_num": meta.get("page_num"),
            "section_title": meta.get("section_title"),
            "file_type": meta.get("file_type"),
            "chunk_id": meta.get("chunk_id"),
        }
        blocks.append(
            f"[문서 {i} | 메타데이터] {json.dumps(head, ensure_ascii=False)}\n"
            f"{d.page_content}"
        )
    return "\n\n" + ("\n\n" + "=" * 70 + "\n\n").join(blocks)

def answer_with_docs(question: str, docs: List[Document], system_name: str = "RAG") -> Dict[str, Any]:
    context = docs_to_context(docs)

    prompt = f"""
당신은 문서기반 질의응답 도우미입니다.
반드시 제공된 문서 내용에 근거해서만 답하세요.
문서에 없는 내용은 추측하지 말고 "문서에서 확인되지 않습니다."라고 말하세요.

[현재 시스템]
{system_name}

[질문]
{question}

[검색 문맥]
{context}

[답변 작성 규칙]
1. 질문에 직접 답할 것
2. 가능하면 근거가 된 문서 번호를 간단히 언급할 것
3. 문서 밖 상식 추론은 최소화할 것
4. 한국어로 답할 것
""".strip()

    response = llm.invoke(prompt).content

    return {
        "question": question,
        "answer": response,
        "contexts": [d.page_content for d in docs],
        "docs": docs,
        "system_name": system_name,
    }

def run_retriever_qa(retriever, question: str, system_name: str) -> Dict[str, Any]:
    docs = retriever.invoke(question)
    return answer_with_docs(question, docs, system_name=system_name)

print("공통 유틸리티 준비 완료")

공통 유틸리티 준비 완료


In [5]:
# ============================================================
# [CELL 4] 원문 로드 + 메타데이터 강화
# - 1단계에서 말한 "메타데이터 강화"를 가장 먼저 적용합니다.
# - 이 단계는 retrieval 전에 문서를 더 잘 설명하는 부가정보를 붙이는 작업입니다.
# - 날짜, 파일타입, 페이지, 섹션 제목, 키워드, chunk id 같은 정보는
#   이후 metadata filtering / router / 디버깅 / 평가 해석에 매우 유용합니다.
# ============================================================

raw_docs = load_documents(DOCUMENT_PATH)
base_docs = enrich_metadata(raw_docs, DOCUMENT_PATH)

print("원문 문서 수:", len(raw_docs))
print("메타데이터 강화 후 문서 수:", len(base_docs))
print("\n샘플 metadata:")
print(base_docs[0].metadata)
print("\n샘플 content 앞부분:")
print(base_docs[0].page_content[:500])

원문 문서 수: 182
메타데이터 강화 후 문서 수: 182

샘플 metadata:
{'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': './data/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1', 'source_path': './data/Demian.pdf', 'source_file': 'Demian.pdf', 'file_type': 'pdf', 'doc_id': 0, 'section_title': 'DEMIAN', 'keywords': ['demian', 'downloaded', 'https', 'www', 'holybooks', 'com'], 'page_num': 0}

샘플 content 앞부분:
DEMIAN 
• 
Downloaded from https://www.holybooks.com


### 1단계. Naive RAG
- OpenAI 기반 기본형

In [11]:
# ============================================================
# [CELL 5] 1단계 - Naive RAG용 기본 청킹 / 벡터DB
# - 가장 단순한 baseline 
#   RecursiveCharacterTextSplitter + Chroma + dense retriever 만 사용
# ============================================================

naive_chunks = make_recursive_chunks(
    base_docs,
    chunk_size=500,
    chunk_overlap=0
)

naive_db = build_chroma(naive_chunks, "naive")
naive_retriever = naive_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

print("Naive chunk 수:", len(naive_chunks))

Naive chunk 수: 700


In [13]:
# ============================================================
# [CELL 6] 1단계 - Naive RAG 테스트
# - baseline이 실제로 어떻게 답하는지 확인
# -이후 단계에서 품질 차이를 체감
# ============================================================

test_question = "이 문서의 핵심 주제는 무엇인가요?"

naive_result = run_retriever_qa(
    naive_retriever,
    test_question,
    system_name="0단계 Naive RAG"
)

print(naive_result["answer"])

이 문서의 핵심 주제는 개인의 독특한 경험과 정체성, 그리고 사회와의 관계에 대한 탐구입니다. 특히, 각 개인이 세상에서 중요한 교차점으로서의 역할을 한다는 점이 강조되고 있습니다. (문서 1)


### 2단계. Naive RAG에 추가하기 쉬운 것
- 메타데이터 강화
- Semantic Chunking
- Query Rewrite / Multi-Query
- Re-ranking
- Context Compression

In [17]:
# ============================================================
# [CELL 7] 2단계 - Semantic Chunking
# - 고정 길이 chunking 대신 의미 단위로 자르기 위한 것
# - Naive RAG 기본 splitter를 유지하되, 비교용으로 semantic split 버전을 별도로 추가
# - 긴 문서에서 주제가 바뀌는 경계가 분명할수록 효과를 보기 수월
# ============================================================

semantic_chunks = make_semantic_chunks(base_docs)
semantic_db = build_chroma(semantic_chunks, "semantic")
semantic_retriever = semantic_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

print("Semantic chunk 수:", len(semantic_chunks))
print(semantic_chunks[0].metadata)
print(semantic_chunks[0].page_content[:500])

Semantic chunk 수: 142
{'chunk_id': 0, 'chunking': 'semantic', 'source_path': './data/Demian.pdf', 'source_file': 'Demian.pdf', 'file_type': 'pdf'}
DEMIAN 
• 
Downloaded from https://www.holybooks.com

HERMANN 
HESSE 
• DEMIAN 
* 
Translated by W. J. Strachan 
London 
Downloaded from https://www.holybooks.com

Prologue 
I cannot tell my story without going a long way back. If it were possible I would go back much farther still to 
the very earliest years of my childhood and beyond them 
to my family origins. When poets write novels they are apt to behave as if 
they were gods, with the power to look beyond and com­
prehend any human story a


In [19]:
# ============================================================
# [CELL 8] 2단계 - Query Rewrite / Multi-Query
# - 사용자의 질문을 여러 관점으로 다시 써서 검색 recall을 높임
# - 질문이 짧거나 모호할 때, 하나의 query만 쓰는 것보다 여러 query union이 유리할 수 있음.
# - semantic_db를 바탕으로 MultiQueryRetriever를 붙임
# ============================================================

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=semantic_db.as_retriever(search_kwargs={"k": TOP_K}),
    llm=llm
)

mq_result = run_retriever_qa(
    multi_query_retriever,
    test_question,
    system_name="1단계 Semantic + MultiQuery"
)

print(mq_result["answer"])

이 문서의 핵심 주제는 개인의 자유 의지와 사회의 금기에 대한 탐구입니다. 등장인물들은 '허용된' 것과 '금지된' 것의 의미를 논의하며, 각자가 스스로의 판단으로 무엇이 허용되고 금지되는지를 발견해야 한다고 강조합니다. 또한, 꿈과 내면의 갈등, 그리고 새로운 세계의 탄생에 대한 예감도 중요한 요소로 나타납니다. (문서 3, 문서 6)


In [21]:
# ============================================================
# [CELL 9] 2단계 - Re-ranking
# - 1차 검색으로 후보 문서를 넓게 가져오고,
#   cross-encoder reranker로 질문-문서 쌍을 다시 정밀하게 재정렬
# - dense retrieval이 잘 찾은 후보들 중 "진짜로 질문에 맞는 순서"를 다시 잡아주는 역할
# - top_n을 줄이면 precision이 오를 수 있지만, 너무 낮추면 recall이 줄 수 있음
# ============================================================

rerank_base_retriever = semantic_db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": TOP_K, "fetch_k": FETCH_K}
)

cross_encoder_model = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-base"
)

reranker = CrossEncoderReranker(
    model=cross_encoder_model,
    top_n=TOP_K
)

rerank_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=rerank_base_retriever
)

rerank_result = run_retriever_qa(
    rerank_retriever,
    test_question,
    system_name="1단계 Semantic + Re-ranking"
)

print(rerank_result["answer"])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


이 문서의 핵심 주제는 자기 인식과 개인의 내면적 발전에 관한 것입니다. 특히, 'Abraxas'라는 개념이 자기 지식으로 나아가는 중요한 단계로 언급되며, 이는 개인의 진화적 과정과 관련이 있습니다 (문서 1).


In [23]:
# ============================================================
# [CELL 10] 2단계 - Context Compression
# - reranking이 "문서 순서"를 정리한다면,
#   compression은 "문서 안에서 질문과 무관한 부분을 줄이는" 역할에 가까움
# - ContextualCompressionRetriever를 다시 사용하되,
#   같은 reranker를 통해 상위 문맥만 남기는 방식으로 간단히 구성
# - 실무에서는 token 절약과 noise 감소 측면에서 실용적
# ============================================================

compression_base_retriever = semantic_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": FETCH_K}
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=compression_base_retriever
)

compression_result = run_retriever_qa(
    compression_retriever,
    test_question,
    system_name="1단계 Semantic + Context Compression"
)

print(compression_result["answer"])

이 문서의 핵심 주제는 개인의 자아 발견과 사회의 규범에 대한 비판적 성찰입니다. 주인공은 '허용된' 세계와 '금지된' 세계의 경계를 탐구하며, 각 개인이 스스로의 가치와 진리를 찾아야 한다는 메시지를 전달합니다. 또한, 인간의 본성과 새로운 가능성에 대한 탐구가 강조됩니다. (문서 1, 4)


### 3단계. 성능 실험용(리서치/확장)
- HyDE
- Hybrid Retrieval

In [26]:
# ============================================================
# [CELL 11] 3단계 - HyDE
# - HyDE(Hypothetical Document Embeddings)는
#   먼저 "가상의 이상적인 답변/문서"를 만들고,
#   그 가상 문서를 query처럼 사용하여 retrieval 성능을 높이는 방식
# - 질문이 직접적인 키워드와 맞지 않을 때 성능이 좋아질 수 있음
# - 여기서는 retriever를 클래스로 만들지 않고 함수형으로 단순하게 구현
# ============================================================

def hyde_retrieve(question: str, base_retriever, n_docs: int = TOP_K) -> List[Document]:
    hyde_prompt = f"""
당신은 검색 질의 확장 도우미입니다.
아래 질문에 답하기 위해, 검색에 유리한 "가상의 짧은 문서"를 4~6문장으로 작성하세요.
추측성 장문이 아니라 핵심 키워드와 개념이 잘 드러나야 합니다.

질문:
{question}
""".strip()

    hypothetical_doc = llm.invoke(hyde_prompt).content
    docs = base_retriever.invoke(hypothetical_doc)
    return docs[:n_docs]

def run_hyde_qa(question: str) -> Dict[str, Any]:
    docs = hyde_retrieve(
        question,
        semantic_db.as_retriever(search_kwargs={"k": FETCH_K}),
        n_docs=TOP_K
    )
    return answer_with_docs(question, docs, system_name="2단계 HyDE")

hyde_result = run_hyde_qa(test_question)
print(hyde_result["answer"])

이 문서의 핵심 주제는 개인의 자아 발견과 사회의 집단적 본능에 대한 비판입니다. 문서에서는 개인이 자신을 이해하고 개별성을 찾는 것이 중요하다고 강조하며, 현재 사회가 두려움과 불안에 기반한 집단적 사고에 갇혀 있다는 점을 지적합니다. 또한, 새로운 이상과 가능성을 추구하는 개인의 역할이 강조됩니다. (문서 1, 3)


In [28]:
# ============================================================
# [CELL 12] 3단계 - Hybrid Retrieval
# - dense(임베딩) 검색과 sparse(BM25) 검색을 함께 사용
# - dense는 의미 유사성에 강하고,
#   sparse는 정확한 키워드/고유명사/숫자 매칭에 강함
# - 둘을 EnsembleRetriever로 합치면 recall/precision 균형이 좋아지는 경우가 많음
# ============================================================

# Dense retriever
dense_retriever = semantic_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

# Sparse retriever
bm25_retriever = BM25Retriever.from_documents(semantic_chunks)
bm25_retriever.k = TOP_K

# Hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6]
)

hybrid_result = run_retriever_qa(
    hybrid_retriever,
    test_question,
    system_name="2단계 Hybrid Retrieval"
)

print(hybrid_result["answer"])

이 문서의 핵심 주제는 개인의 내면적 성장과 자아 발견, 그리고 인간 존재의 의미에 대한 탐구입니다. 특히, '데미안'이라는 인물과의 관계를 통해 주인공이 자신의 정체성과 삶의 목적을 찾아가는 과정을 다루고 있습니다. 문서에서는 고대 이야기와 신화, 인간의 본성과 의지에 대한 철학적 논의도 포함되어 있습니다. (문서 2, 4, 6, 8 참조)


In [30]:
# ============================================================
# [CELL 13] 3단계 - 데이터 타입별 분기
# - 문서 형식이 PDF / TXT / CSV일 때 retrieval 전략을 다르게 줄 수 있도록 간단한 router 성격을 먼저 실험하는 단계
# - 예:
#   * PDF: semantic + rerank
#   * TXT/MD: semantic + multi-query
#   * CSV: hybrid retrieval
# - 실제 서비스에서는 테이블형 데이터, 보고서형 데이터, FAQ형 데이터를 분기하는 데 유용함
# ============================================================

def get_type_based_retriever(file_type: str):
    file_type = file_type.lower()

    if file_type == "pdf":
        return rerank_retriever
    elif file_type in ["txt", "md"]:
        return multi_query_retriever
    elif file_type == "csv":
        return hybrid_retriever
    else:
        return naive_retriever

type_based_retriever = get_type_based_retriever(detect_file_type(DOCUMENT_PATH))

type_branch_result = run_retriever_qa(
    type_based_retriever,
    test_question,
    system_name="2단계 데이터 타입별 분기"
)

print(type_branch_result["answer"])

이 문서의 핵심 주제는 자기 인식과 개인의 발전에 관한 내용입니다. 특히, 'Abraxas'라는 개념이 자기 지식의 진전을 의미한다는 점이 강조되고 있습니다(문서 1).


### 3-2단계. 고급 확장
- Self-RAG 축소판
- Iterative Retrieval
- Router 기반 Modular RAG

In [33]:
# ============================================================
# [CELL 14] 3-2단계 - Self-RAG 축소판
# - 완전한 Self-RAG 구현은 아니고, notebook 실습용 축소판
# - 흐름:
#   1) 1차 검색 후 답변 생성
#   2) LLM이 "근거 충분성 / 답변 신뢰도"를 스스로 점검
#   3) 부족하면 더 넓게 재검색해서 2차 답변 생성
# - 핵심은 "한 번 답하고 끝"이 아니라 "답변 전에 스스로 검토"하는 루프를 넣음
# ============================================================

def self_rag_lite(question: str) -> Dict[str, Any]:
    # 1차 검색
    first_docs = hybrid_retriever.invoke(question)
    first_answer = answer_with_docs(
        question,
        first_docs,
        system_name="3단계 Self-RAG-lite (1차)"
    )["answer"]

    critique_prompt = f"""
아래 답변이 제공 문맥만으로 충분히 뒷받침되는지 평가하세요.

[질문]
{question}

[1차 답변]
{first_answer}

판정 형식:
- verdict: SUFFICIENT 또는 INSUFFICIENT
- reason: 한 줄 설명
""".strip()

    critique = llm.invoke(critique_prompt).content

    if "INSUFFICIENT" in critique.upper():
        # 2차 검색: 더 넓게 + multi query 성격 반영
        expanded_query_prompt = f"""
아래 질문을 더 잘 검색되게 3개의 검색 질의로 재작성하세요.
질문: {question}
""".strip()

        expanded_queries = llm.invoke(expanded_query_prompt).content
        second_docs = hybrid_retriever.invoke(expanded_queries)
        second_answer_dict = answer_with_docs(
            question,
            second_docs,
            system_name="3단계 Self-RAG-lite (재검색)"
        )
        second_answer_dict["critique"] = critique
        return second_answer_dict

    result = {
        "question": question,
        "answer": first_answer,
        "contexts": [d.page_content for d in first_docs],
        "docs": first_docs,
        "system_name": "3단계 Self-RAG-lite",
        "critique": critique
    }
    return result

self_rag_result = self_rag_lite(test_question)
print(self_rag_result["answer"])
print("\n[critique]")
print(self_rag_result["critique"])

이 문서의 핵심 주제는 인간의 내면과 자아의 탐구, 그리고 개인의 의지와 운명에 대한 성찰입니다. 특히, '자유 의지'와 '운명'의 관계, 그리고 인간 존재의 깊은 의미를 탐구하는 내용이 포함되어 있습니다. 예를 들어, 문서 2에서는 주인공이 Max Demian과의 대화를 통해 성경의 이야기인 카인의 해석을 다르게 바라보는 시각을 배우고, 문서 4에서는 개인의 경험과 존재의 의미에 대한 철학적 논의가 이루어집니다. 이러한 주제들은 주인공의 성장과 자아 발견의 과정을 통해 드러납니다.

[critique]
- verdict: SUFFICIENT  
- reason: 제공된 문맥에서 문서의 핵심 주제와 관련된 구체적인 내용과 예시가 충분히 설명되어 있어 주제를 명확히 이해할 수 있다.


In [35]:
# ============================================================
# [CELL 15] 3-2단계 - Iterative Retrieval
# - 질문 -> 검색 -> 임시답변 -> 부족한 정보 식별 -> 보강 검색
#   구조를 2회전 정도만 돌리는 간단한 버전
# - 복잡한 질문일수록 한 번에 끝내지 않고,
#   "무엇이 아직 부족한가?"를 다시 질의로 바꿈
# ============================================================

def iterative_retrieval(question: str, max_rounds: int = 2) -> Dict[str, Any]:
    current_query = question
    collected_docs = []

    for round_idx in range(max_rounds):
        docs = hybrid_retriever.invoke(current_query)
        collected_docs.extend(docs)

        # 중복 제거
        unique_docs = []
        seen = set()
        for d in collected_docs:
            key = (d.page_content[:200], json.dumps(d.metadata, ensure_ascii=False, sort_keys=True))
            if key not in seen:
                seen.add(key)
                unique_docs.append(d)
        collected_docs = unique_docs[:8]

        draft = answer_with_docs(
            question,
            collected_docs,
            system_name=f"3단계 Iterative Retrieval (round {round_idx+1})"
        )["answer"]

        followup_prompt = f"""
원 질문에 더 잘 답하기 위해 추가로 검색해야 할 부족 정보가 있으면
짧은 검색 질의 한 줄만 작성하세요.
더 이상 필요 없으면 DONE만 출력하세요.

[원 질문]
{question}

[현재 초안]
{draft}
""".strip()

        next_query = llm.invoke(followup_prompt).content.strip()

        if next_query.upper() == "DONE":
            return {
                "question": question,
                "answer": draft,
                "contexts": [d.page_content for d in collected_docs],
                "docs": collected_docs,
                "system_name": "3단계 Iterative Retrieval"
            }

        current_query = next_query

    final = answer_with_docs(
        question,
        collected_docs,
        system_name="3단계 Iterative Retrieval"
    )
    return final

iter_result = iterative_retrieval(test_question)
print(iter_result["answer"])

이 문서의 핵심 주제는 인간의 내면과 자아의 탐구, 그리고 개인의 의지와 정체성에 대한 성찰입니다. 특히, '데미안'이라는 인물과의 관계를 통해 주인공이 자신의 존재와 삶의 의미를 찾고, 전통적인 가치관과 신념에 도전하는 과정을 다루고 있습니다. 이러한 주제는 문서 전반에 걸쳐 나타나며, 특히 문서 2와 4에서 두드러지게 나타납니다.


In [37]:
# ============================================================
# [CELL 16] 3-2단계 - Router 기반 Modular RAG
# - 하나의 retriever만 쓰지 않고,
#   질문 유형에 따라 어떤 retrieval 전략을 쓸지 라우팅하는 단계
# - 여기서는 너무 복잡한 classifier 대신,
#   LLM + 간단 규칙 기반 router로 구성함
# - 라우팅 대상:
#   1) factual / keyword 중심 -> hybrid
#   2) broad / abstract 질문 -> multi-query
#   3) difficult / vague 질문 -> HyDE
#   4) page/section/source 의식 질문 -> rerank
# ============================================================

def route_question(question: str) -> str:
    lower_q = question.lower()

    # 간단한 규칙 우선
    if any(k in lower_q for k in ["page", "페이지", "section", "챕터", "장", "source", "출처"]):
        return "rerank"

    router_prompt = f"""
아래 질문을 가장 적절한 검색 전략 하나로 분류하세요.

전략 후보:
- hybrid : 키워드/고유명사/숫자/팩트 중심 질문
- multiquery : 넓고 설명형이며 관점 확장이 필요한 질문
- hyde : 질문이 모호하거나 직접 검색어가 잘 안 나올 것 같은 질문
- rerank : 이미 후보는 잘 잡히지만 순서 정밀화가 중요한 질문

질문:
{question}

정답은 전략 이름 하나만 출력하세요.
""".strip()

    route = llm.invoke(router_prompt).content.strip().lower()

    if "hybrid" in route:
        return "hybrid"
    elif "multi" in route:
        return "multiquery"
    elif "hyde" in route:
        return "hyde"
    else:
        return "rerank"

def modular_rag(question: str) -> Dict[str, Any]:
    route = route_question(question)

    if route == "hybrid":
        result = run_retriever_qa(hybrid_retriever, question, "3단계 Router Modular RAG - hybrid")
    elif route == "multiquery":
        result = run_retriever_qa(multi_query_retriever, question, "3단계 Router Modular RAG - multiquery")
    elif route == "hyde":
        result = run_hyde_qa(question)
        result["system_name"] = "3단계 Router Modular RAG - hyde"
    else:
        result = run_retriever_qa(rerank_retriever, question, "3단계 Router Modular RAG - rerank")

    result["route"] = route
    return result

router_result = modular_rag(test_question)
print("route:", router_result["route"])
print(router_result["answer"])

route: multiquery
이 문서의 핵심 주제는 개인의 자유 의지와 도덕적 선택에 대한 탐구입니다. 등장인물들은 '허용된' 것과 '금지된' 것의 개념을 논의하며, 각자가 스스로의 판단에 따라 무엇이 허용되고 금지되는지를 발견해야 한다고 강조합니다. 또한, 꿈과 내면의 갈등, 그리고 개인의 성장 과정에 대한 이야기도 포함되어 있습니다. (문서 3, 6)


### 4단계. 평가
- Faithfulness: 생성된 답변이 검색된 문맥(retrieved contexts)에 근거해 충실하게 작성되었는가
- Answer Relevancy: 답변이 사용자의 질문 의도에 적절하게 답하고 있는가
- Context Precision: 검색된 문맥 중 실제로 답변에 유효한 근거의 비율이 얼마나 높은가
- Context Recall: 정답에 필요한 근거를 검색 단계에서 얼마나 빠짐없이 가져왔는가

- 이번 평가에서는 RAGAS 지표를 사용해 각 시스템의 답변 품질을 비교한다.
- 평가 품질을 높이기 위해 추상적인 reference 대신 문서 기반의 구체적인 기준답(reference)과 필수 포인트(required_points)를 함께 구성한다.
- 또한 retrieved contexts를 정리하고, row-level 평가 후 system-level 평균으로 집계하여 어떤 방식이 더 안정적인지 비교한다.
- 추가로 G-EVAL을 5회 반복 수행하여 정성 평가를 병행한다.

In [42]:
# ============================================================
# [CELL 17] 4단계 평가: 비교용 시스템 레지스트리 (평가 안정화 버전)
# - 목적:
#   1) 여러 RAG 시스템을 동일한 인터페이스로 등록
#   2) 각 시스템의 반환 형식을 강제로 표준화
#   3) 이후 CELL 18+에서 RAGAS / G-EVAL 평가를 안정적으로 수행
#
# - 표준 반환 형식:
#   {
#       "answer": str,
#       "contexts": List[str],
#       "raw_result": Any
#   }
#
# - 왜 필요한가:
#   * 어떤 함수는 {"answer": ...}를 반환하고
#   * 어떤 함수는 {"result": ...}, {"final_answer": ...}를 반환할 수 있음
#   * 어떤 함수는 contexts 대신 docs / retrieved_docs / source_documents 등을 쓸 수 있음
#   * 평가 단계에서는 answer / contexts 키가 통일되어야 함
# ============================================================

from typing import Any, Dict, List

# -----------------------------
# 1) Document / 문자열 / 기타 타입을
#    평가용 context 문자열 리스트로 통일하는 helper
# -----------------------------
def _to_context_text_list(value: Any) -> List[str]:
    """
    입력이 무엇이든 최종적으로 List[str] 형태의 contexts로 변환합니다.

    처리 대상 예시:
    - List[Document]
    - List[str]
    - 단일 Document
    - 단일 str
    - None
    """
    if value is None:
        return []

    # 단일 문자열
    if isinstance(value, str):
        text = value.strip()
        return [text] if text else []

    # LangChain Document 비슷한 객체
    if hasattr(value, "page_content"):
        text = getattr(value, "page_content", "")
        text = text.strip() if isinstance(text, str) else str(text).strip()
        return [text] if text else []

    # 리스트/튜플
    if isinstance(value, (list, tuple)):
        out = []
        for item in value:
            if item is None:
                continue
            if isinstance(item, str):
                text = item.strip()
                if text:
                    out.append(text)
            elif hasattr(item, "page_content"):
                text = getattr(item, "page_content", "")
                text = text.strip() if isinstance(text, str) else str(text).strip()
                if text:
                    out.append(text)
            else:
                text = str(item).strip()
                if text:
                    out.append(text)
        return out

    # 기타 타입
    text = str(value).strip()
    return [text] if text else []

# -----------------------------
# 2) 시스템별 원본 결과를 표준 포맷으로 통일하는 helper
# -----------------------------
def standardize_result(raw_result: Any) -> Dict[str, Any]:
    """
    다양한 반환 형식을 아래 표준 형식으로 변환합니다.

    표준 형식:
    {
        "answer": str,
        "contexts": List[str],
        "raw_result": raw_result
    }
    """
    answer = ""
    contexts = []

    # case 1) dict 반환
    if isinstance(raw_result, dict):
        # answer 후보 키 우선순위
        answer_candidates = [
            "answer",
            "final_answer",
            "result",
            "output_text",
            "response",
        ]
        for key in answer_candidates:
            if key in raw_result and raw_result[key] is not None:
                answer = str(raw_result[key]).strip()
                if answer:
                    break

        # contexts 후보 키 우선순위
        context_candidates = [
            "contexts",
            "docs",
            "retrieved_docs",
            "source_documents",
            "documents",
            "context",
        ]
        for key in context_candidates:
            if key in raw_result and raw_result[key] is not None:
                contexts = _to_context_text_list(raw_result[key])
                if contexts:
                    break

        # 만약 dict 안에 answer가 끝까지 없으면 전체를 문자열화하지 말고 빈 문자열 유지
        # (평가에서 오류 추적이 더 쉬움)

    # case 2) 문자열 반환
    elif isinstance(raw_result, str):
        answer = raw_result.strip()
        contexts = []

    # case 3) 기타 객체 반환
    else:
        # 혹시 page_content가 있는 단일 Document류면 context로 처리
        if hasattr(raw_result, "page_content"):
            contexts = _to_context_text_list(raw_result)
            answer = ""
        else:
            answer = str(raw_result).strip()

    return {
        "answer": answer,
        "contexts": contexts,
        "raw_result": raw_result,
    }

# -----------------------------
# 3) 개별 실행 함수를 안전하게 감싸는 wrapper
# -----------------------------
def make_standard_runner(run_fn, runner_name: str):
    """
    각 시스템 실행 함수를 감싸서
    항상 표준 포맷 dict를 반환하게 만듭니다.
    """
    def _wrapped(question: str) -> Dict[str, Any]:
        try:
            raw = run_fn(question)
            standardized = standardize_result(raw)
            return standardized
        except Exception as e:
            return {
                "answer": f"[ERROR] {runner_name}: {type(e).__name__}: {str(e)}",
                "contexts": [],
                "raw_result": {
                    "error_type": type(e).__name__,
                    "error_message": str(e),
                    "runner_name": runner_name,
                }
            }
    return _wrapped

# -----------------------------
# 4) 원본 실행 함수들
# - run_retriever_qa 계열은 기존 구조 유지
# - 뒤에서 make_standard_runner()로 감싸서 표준화
# -----------------------------
def run_naive_raw(question: str):
    return run_retriever_qa(naive_retriever, question, "1단계 Naive RAG")

def run_semantic_raw(question: str):
    return run_retriever_qa(semantic_retriever, question, "2단계 Semantic Chunking")

def run_multiquery_raw(question: str):
    return run_retriever_qa(multi_query_retriever, question, "2단계 MultiQuery")

def run_rerank_raw(question: str):
    return run_retriever_qa(rerank_retriever, question, "2단계 Re-ranking")

def run_compression_raw(question: str):
    return run_retriever_qa(compression_retriever, question, "2단계 Context Compression")

def run_hybrid_raw(question: str):
    return run_retriever_qa(hybrid_retriever, question, "3단계 Hybrid Retrieval")

def run_type_branch_raw(question: str):
    return run_retriever_qa(type_based_retriever, question, "3단계 데이터 타입별 분기")

# -----------------------------
# 5) 표준화 wrapper 적용
# -----------------------------
run_naive = make_standard_runner(run_naive_raw, "naive")
run_semantic = make_standard_runner(run_semantic_raw, "semantic")
run_multiquery = make_standard_runner(run_multiquery_raw, "multiquery")
run_rerank = make_standard_runner(run_rerank_raw, "rerank")
run_compression = make_standard_runner(run_compression_raw, "compression")
run_hyde = make_standard_runner(run_hyde_qa, "hyde")
run_hybrid = make_standard_runner(run_hybrid_raw, "hybrid")
run_type_branch = make_standard_runner(run_type_branch_raw, "type_branch")
run_self_rag_lite = make_standard_runner(self_rag_lite, "self_rag_lite")
run_iterative = make_standard_runner(iterative_retrieval, "iterative")
run_router_modular = make_standard_runner(modular_rag, "router_modular")

# -----------------------------
# 6) 시스템 레지스트리
# - 이후 CELL 18+에서 이 registry를 사용해
#   동일 질문셋을 공평하게 비교
# -----------------------------
system_registry = {
    "naive": run_naive,
    "semantic": run_semantic,
    "multiquery": run_multiquery,
    "rerank": run_rerank,
    "compression": run_compression,
    "hyde": run_hyde,
    "hybrid": run_hybrid,
    "type_branch": run_type_branch,
    "self_rag_lite": run_self_rag_lite,
    "iterative": run_iterative,
    "router_modular": run_router_modular,
}

print("등록된 시스템:", list(system_registry.keys()))

# -----------------------------
# 7) 간단한 형식 점검용 테스트
# - 실제 평가 전에 한 번 확인하면 좋습니다.
# - 필요 없으면 주석 처리해도 됩니다.
# -----------------------------
_test_question = "이 문서의 핵심 주제는 무엇인가요?"

for _name, _fn in system_registry.items():
    _result = _fn(_test_question)
    print(f"\n[{_name}]")
    print("answer type :", type(_result.get("answer")).__name__)
    print("contexts type:", type(_result.get("contexts")).__name__)
    print("num_contexts :", len(_result.get("contexts", [])))
    print("answer preview:", str(_result.get("answer", ""))[:120])

등록된 시스템: ['naive', 'semantic', 'multiquery', 'rerank', 'compression', 'hyde', 'hybrid', 'type_branch', 'self_rag_lite', 'iterative', 'router_modular']

[naive]
answer type : str
contexts type: list
num_contexts : 4
answer preview: 이 문서의 핵심 주제는 개인의 독특한 경험과 정체성, 그리고 사회와의 관계에 대한 탐구입니다. 특히, 각 개인이 세상에서 중요한 교차점으로서의 역할을 한다는 점이 강조됩니다. (문서 1)

[semantic]
answer type : str
contexts type: list
num_contexts : 4
answer preview: 이 문서의 핵심 주제는 개인의 내면적 성장과 자아 발견, 그리고 인간 존재의 복잡성에 대한 탐구입니다. 특히, 문서에서는 '자유 의지'와 '의지의 힘', 그리고 고대 이야기의 해석을 통해 인간의 본성과 잠재력에 대한

[multiquery]
answer type : str
contexts type: list
num_contexts : 8
answer preview: 이 문서의 핵심 주제는 개인의 자유 의지와 사회의 규범, 그리고 개인이 스스로의 삶을 어떻게 이해하고 살아가야 하는지를 탐구하는 것입니다. 특히, '허용된' 것과 '금지된' 것의 개념을 통해 각자가 스스로의 판단을 

[rerank]
answer type : str
contexts type: list
num_contexts : 4
answer preview: 이 문서의 핵심 주제는 자기 인식과 개인의 내면적 발전에 관한 것입니다. 특히, 'Abraxas'라는 개념이 자기 지식으로 나아가는 중요한 단계로 언급되며, 이는 개인의 진화적 과정과 관련이 있습니다(문서 1).

[compression]
answer type : str
contexts type: list
n

In [44]:
# ============================================================
# [CELL 18] 4단계 평가: 평가 공통 유틸리티
# - 목적:
#   1) RAGAS 평가에 넣을 "골든셋(reference)" 품질을 높이기 위한 준비
#   2) 문서 원문을 기반으로 질문/정답/필수포인트를 더 구체적으로 생성
#   3) retrieved_contexts를 정리/중복제거해서 RAGAS 입력 품질 개선
#
# - 핵심 아이디어:
#   * 기존 reference = "문서의 핵심 주제를 간결하게 요약한 답변"
#     -> 너무 추상적이라 RAGAS 품질이 흔들림
#   * 따라서 먼저 문서 기반 gold sample(question/reference/required_points)을
#     더 구체적으로 만든 뒤 평가에 사용
# ============================================================

import json
import re
import math
import asyncio
import statistics
from copy import deepcopy
from typing import List, Dict, Any, Callable

import pandas as pd

# -----------------------------
# 1) 긴 문서를 골든셋 생성용으로 압축하는 helper
# -----------------------------
def normalize_whitespace(text: str) -> str:
    text = text or ""
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def truncate_text(text: str, max_chars: int = 1200) -> str:
    text = normalize_whitespace(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + " ..."

def build_gold_source_packets(
    docs: List[Document],
    max_docs: int = 20,
    max_chars_per_doc: int = 1200
) -> str:
    """
    골든셋 생성을 위해 문서 내용을 너무 길지 않게 묶어서 LLM에 전달합니다.
    page_num / source_file / section_title 같은 메타데이터를 유지해서
    reference가 더 구체적으로 생성되도록 유도합니다.
    """
    packets = []

    use_docs = docs[:max_docs]
    for i, d in enumerate(use_docs, start=1):
        meta = d.metadata or {}
        packet = {
            "idx": i,
            "source_file": meta.get("source_file"),
            "page_num": meta.get("page_num"),
            "section_title": meta.get("section_title"),
            "file_type": meta.get("file_type"),
            "text": truncate_text(d.page_content, max_chars=max_chars_per_doc),
        }
        packets.append(packet)

    return json.dumps(packets, ensure_ascii=False, indent=2)

# -----------------------------
# 2) JSON 파싱 보조 함수
# -----------------------------
def safe_json_loads(text: str):
    """
    LLM 응답이 ```json ... ``` 형태일 수도 있어서 방어적으로 파싱합니다.
    """
    text = text.strip()

    # fenced code block 제거
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    return json.loads(text)

# -----------------------------
# 3) retrieved contexts 정리
# -----------------------------
def clean_contexts(
    contexts: List[str],
    min_chars: int = 30,
    max_per_context: int = 1800,
    max_contexts: int = 6
) -> List[str]:
    """
    RAGAS용 retrieved_contexts 품질 개선:
    - 빈 문자열 제거
    - 너무 짧은 문맥 제거
    - 중복 제거
    - 각 context 길이 과도하면 잘라냄
    """
    cleaned = []
    seen = set()

    for c in contexts or []:
        c = normalize_whitespace(c)
        if len(c) < min_chars:
            continue

        c = truncate_text(c, max_chars=max_per_context)

        # 앞부분 기준으로 중복 제거
        key = c[:300]
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(c)

        if len(cleaned) >= max_contexts:
            break

    return cleaned

# -----------------------------
# 4) 골든셋 dataframe 보기 helper
# -----------------------------
def preview_eval_samples(samples: List[Dict[str, Any]]) -> pd.DataFrame:
    view_rows = []
    for s in samples:
        view_rows.append({
            "question_type": s.get("question_type"),
            "question": s.get("question"),
            "reference": s.get("reference"),
            "required_points": " / ".join(s.get("required_points", [])),
            "eval_focus": s.get("eval_focus"),
        })
    return pd.DataFrame(view_rows)

In [46]:
# ============================================================
# [CELL 19] 4단계 평가: 고품질 eval_samples(골든셋) 생성
# - 목적:
#   1) 기존의 추상적 reference를 버리고
#      문서 기반의 더 구체적인 reference를 자동 생성
#   2) 질문 유형(question_type), 필수포인트(required_points), 평가 초점(eval_focus)
#      까지 같이 생성해서 RAGAS/G-EVAL 둘 다 품질 향상
#
# - 사용 방식:
#   * AUTO_BUILD_EVAL_SAMPLES = True  -> 문서 기반 자동 생성
#   * AUTO_BUILD_EVAL_SAMPLES = False -> 아래 MANUAL_EVAL_SAMPLES를 직접 작성해 사용
#
# - 매우 중요:
#   * 자동 생성 결과는 "초안"입니다.
#   * 실제 제출/보고용 평가에서는 반드시 사람이 한 번 검토하는 것이 가장 좋습니다.
# ============================================================

AUTO_BUILD_EVAL_SAMPLES = True

MANUAL_EVAL_SAMPLES = [
    # 필요하면 여기에 직접 수기로 넣어도 됩니다.
    # 예시 형식:
    # {
    #     "question_type": "summary",
    #     "question": "이 문서의 핵심 주제는 무엇인가요?",
    #     "reference": "문서는 ...을 중심으로 전개되며, ...를 강조한다.",
    #     "required_points": ["핵심 주제 언급", "주요 갈등 언급", "문서 전반 메시지 언급"],
    #     "eval_focus": "핵심 주제와 전개를 문서 근거 기반으로 요약했는가"
    # }
]

def generate_eval_samples_from_document(
    docs: List[Document],
    n_questions: int = 8,
    max_docs: int = 20,
    max_chars_per_doc: int = 1200
) -> List[Dict[str, Any]]:
    """
    문서 기반으로 평가용 골든셋을 생성합니다.
    질문 유형을 섞어서 생성하도록 유도합니다.
    """
    source_packets = build_gold_source_packets(
        docs=docs,
        max_docs=max_docs,
        max_chars_per_doc=max_chars_per_doc
    )

    prompt = f"""
당신은 RAG 평가용 골든셋 설계자입니다.
아래 문서 묶음을 바탕으로 RAG 평가용 질문셋을 JSON 배열로 생성하세요.

[목표]
- 너무 추상적인 기준답이 아니라, 실제로 평가 가능한 구체적 기준답(reference)을 작성
- 질문 유형을 다양하게 섞을 것:
  1) summary
  2) factual
  3) theme
  4) character_or_entity
  5) reasoning_light
  6) tone_or_message
- 각 항목에는 다음 필드를 반드시 포함:
  - question_type
  - question
  - reference
  - required_points   (문자열 리스트 2~4개)
  - eval_focus

[중요 규칙]
- reference는 "문서에서 확인 가능한 내용"만 써야 함
- 모호한 표현 대신 핵심 사실/주제/전개를 분명하게 쓸 것
- 질문은 한국어로 작성
- reference도 한국어로 작성
- required_points는 실제 채점 가능한 짧은 문구로 작성
- 출력은 JSON 배열만 출력

[문서 묶음]
{source_packets}

[생성 개수]
{n_questions}
""".strip()

    raw = llm.invoke(prompt).content
    samples = safe_json_loads(raw)

    # 최소한의 후처리
    normalized = []
    for item in samples:
        normalized.append({
            "question_type": item.get("question_type", "unknown"),
            "question": normalize_whitespace(item.get("question", "")),
            "reference": normalize_whitespace(item.get("reference", "")),
            "required_points": [
                normalize_whitespace(x) for x in item.get("required_points", [])
                if normalize_whitespace(x)
            ],
            "eval_focus": normalize_whitespace(item.get("eval_focus", "")),
        })

    return normalized

if AUTO_BUILD_EVAL_SAMPLES:
    eval_samples = generate_eval_samples_from_document(
        docs=base_docs,
        n_questions=8,
        max_docs=min(len(base_docs), 20),
        max_chars_per_doc=1200
    )
else:
    eval_samples = deepcopy(MANUAL_EVAL_SAMPLES)

print("평가용 샘플 수:", len(eval_samples))
preview_eval_samples(eval_samples)

평가용 샘플 수: 8


,question_type,question,reference,required_points,eval_focus
0,summary,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,"주인공은 자신의 이야기가 시인들의 이야기보다 더 중요하다고 느끼며, 자신의 독특한 ...",자신의 이야기의 중요성 / 독특한 존재에 대한 언급,주인공의 이야기 시작 이유 이해
1,factual,주인공이 다니던 학교는 어떤 종류의 학교였나요?,주인공은 작은 시골 마을의 문법 학교에 다녔다.,작은 시골 마을 / 문법 학교,학교에 대한 정확한 정보
2,theme,이 이야기에서 '두 세계'의 주제는 무엇을 의미하나요?,"주인공은 부모의 집과 그 외부의 두 가지 상반된 세계를 경험하며, 각 세계의 특성을...",부모의 집 / 외부 세계의 대조,주제의 이해와 해석
3,character_or_entity,프란츠 크로머는 어떤 인물인가요?,"프란츠 크로머는 주인공에게 위협을 가하는 인물로, '다른' 세계의 사람이다.",위협적인 인물 / 다른 세계의 사람,프란츠 크로머의 성격 이해
4,reasoning_light,주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?,주인공은 프란츠가 자신을 고발할까 두려워 시계를 주기로 결심했다.,고발에 대한 두려움 / 시계의 가치,주인공의 결정 과정 이해
5,tone_or_message,이 이야기에서 주인공의 감정은 어떤가요?,"주인공은 두 세계 사이에서 혼란과 두려움을 느끼며, 자신의 정체성에 대한 갈등을 겪...",혼란 / 두려움 / 정체성 갈등,주인공의 감정 상태 분석
6,summary,주인공이 어린 시절의 기억을 회상하는 이유는 무엇인가요?,주인공은 자신의 과거를 통해 현재의 자신을 이해하고자 한다.,과거 회상 / 현재 이해,회상의 목적 이해
7,theme,이야기에서 '죄'의 개념은 어떻게 다루어지고 있나요?,"주인공은 자신의 행동이 죄가 될 수 있음을 인식하고, 그로 인해 내적 갈등을 겪고 있다.",죄의 인식 / 내적 갈등,죄의 개념에 대한 이해


In [48]:
# ============================================================
# [CELL 20] 4단계 평가: 시스템별 예측 결과 생성
# - 목적:
#   1) 같은 질문셋을 각 RAG 시스템에 동일하게 실행
#   2) answer / retrieved_contexts / reference / required_points를 함께 보관
#   3) 이후 RAGAS와 G-EVAL 둘 다 같은 eval_df를 사용
#
# - 품질 개선 포인트:
#   * contexts를 clean_contexts()로 정리해서 RAGAS 입력 품질 개선
#   * question_type / eval_focus / required_points도 같이 저장
# ============================================================

def build_eval_rows(
    system_name: str,
    run_fn: Callable,
    samples: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    rows = []

    for item in samples:
        q = item["question"]
        ref = item["reference"]
        question_type = item.get("question_type", "unknown")
        required_points = item.get("required_points", [])
        eval_focus = item.get("eval_focus", "")

        try:
            result = run_fn(q)
            answer_text = result.get("answer", "")
            raw_contexts = result.get("contexts", [])
            cleaned_contexts = clean_contexts(raw_contexts)

            row = {
                # -------- RAGAS 신버전 호환 컬럼 --------
                "user_input": q,
                "response": answer_text,
                "retrieved_contexts": cleaned_contexts,
                "reference": ref,

                # -------- RAGAS 구버전 호환 컬럼 --------
                "question": q,
                "answer": answer_text,
                "contexts": cleaned_contexts,
                "ground_truth": ref,

                # -------- 보조 메타데이터 --------
                "system": system_name,
                "question_type": question_type,
                "required_points": required_points,
                "required_points_text": " / ".join(required_points),
                "eval_focus": eval_focus,
                "num_contexts": len(cleaned_contexts),
                "answer_len": len(normalize_whitespace(answer_text)),
                "error": None,
            }

        except Exception as e:
            row = {
                "user_input": q,
                "response": f"[ERROR] {type(e).__name__}: {str(e)}",
                "retrieved_contexts": [],
                "reference": ref,

                "question": q,
                "answer": f"[ERROR] {type(e).__name__}: {str(e)}",
                "contexts": [],
                "ground_truth": ref,

                "system": system_name,
                "question_type": question_type,
                "required_points": required_points,
                "required_points_text": " / ".join(required_points),
                "eval_focus": eval_focus,
                "num_contexts": 0,
                "answer_len": 0,
                "error": f"{type(e).__name__}: {str(e)}",
            }

        rows.append(row)

    return rows

# -----------------------------
# 비교할 시스템 목록
# -----------------------------
systems_to_compare = [
    "naive",
    "semantic",
    "multiquery",
    "rerank",
    "compression",
    "hyde",
    "hybrid",
    "self_rag_lite",
    "iterative",
    "router_modular",
]

all_eval_rows = []

for name in systems_to_compare:
    print(f"Running: {name}")
    rows = build_eval_rows(name, system_registry[name], eval_samples)
    all_eval_rows.extend(rows)

eval_df = pd.DataFrame(all_eval_rows)

print("eval_df shape:", eval_df.shape)
display(
    eval_df[[
        "system", "question_type", "question", "reference",
        "required_points_text", "num_contexts", "answer_len", "error"
    ]].head(20)
)

Running: naive
Running: semantic
Running: multiquery
Running: rerank
Running: compression
Running: hyde
Running: hybrid
Running: self_rag_lite
Running: iterative
Running: router_modular
eval_df shape: (80, 16)


,system,question_type,question,reference,required_points_text,num_contexts,answer_len,error
0,naive,summary,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,"주인공은 자신의 이야기가 시인들의 이야기보다 더 중요하다고 느끼며, 자신의 독특한 ...",자신의 이야기의 중요성 / 독특한 존재에 대한 언급,4,132,None
1,naive,factual,주인공이 다니던 학교는 어떤 종류의 학교였나요?,주인공은 작은 시골 마을의 문법 학교에 다녔다.,작은 시골 마을 / 문법 학교,4,35,None
2,naive,theme,이 이야기에서 '두 세계'의 주제는 무엇을 의미하나요?,"주인공은 부모의 집과 그 외부의 두 가지 상반된 세계를 경험하며, 각 세계의 특성을...",부모의 집 / 외부 세계의 대조,4,191,None
3,naive,character_or_entity,프란츠 크로머는 어떤 인물인가요?,"프란츠 크로머는 주인공에게 위협을 가하는 인물로, '다른' 세계의 사람이다.",위협적인 인물 / 다른 세계의 사람,4,31,None
4,naive,reasoning_light,주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?,주인공은 프란츠가 자신을 고발할까 두려워 시계를 주기로 결심했다.,고발에 대한 두려움 / 시계의 가치,4,46,None
5,naive,tone_or_message,이 이야기에서 주인공의 감정은 어떤가요?,"주인공은 두 세계 사이에서 혼란과 두려움을 느끼며, 자신의 정체성에 대한 갈등을 겪...",혼란 / 두려움 / 정체성 갈등,4,155,None
6,naive,summary,주인공이 어린 시절의 기억을 회상하는 이유는 무엇인가요?,주인공은 자신의 과거를 통해 현재의 자신을 이해하고자 한다.,과거 회상 / 현재 이해,4,194,None
7,naive,theme,이야기에서 '죄'의 개념은 어떻게 다루어지고 있나요?,"주인공은 자신의 행동이 죄가 될 수 있음을 인식하고, 그로 인해 내적 갈등을 겪고 있다.",죄의 인식 / 내적 갈등,4,425,None
8,semantic,summary,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,"주인공은 자신의 이야기가 시인들의 이야기보다 더 중요하다고 느끼며, 자신의 독특한 ...",자신의 이야기의 중요성 / 독특한 존재에 대한 언급,4,207,None
9,semantic,factual,주인공이 다니던 학교는 어떤 종류의 학교였나요?,주인공은 작은 시골 마을의 문법 학교에 다녔다.,작은 시골 마을 / 문법 학교,4,53,None


In [50]:
# ============================================================
# [CELL 21] 4단계 평가: RAGAS metric 준비
# - 목적:
#   1) 기본 RAGAS 4개 지표(answer_relevancy, faithfulness,
#      context_recall, context_precision) 설정
#   2) 가능하면 answer_correctness도 추가
#
# - 품질 개선 포인트:
#   * reference 품질을 높인 상태에서 metric을 돌리므로
#     기존보다 RAGAS 해석력이 좋아짐
# ============================================================

import ragas
print("ragas version:", ragas.__version__)

from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings

from ragas.metrics.collections.answer_relevancy import AnswerRelevancy
from ragas.metrics.collections.faithfulness import Faithfulness
from ragas.metrics.collections.context_recall import ContextRecall
from ragas.metrics.collections.context_precision import ContextPrecision

# answer_correctness는 버전에 따라 위치가 다를 수 있으므로 방어적으로 import
RAGAS_HAS_ANSWER_CORRECTNESS = False
try:
    from ragas.metrics.collections.answer_correctness import AnswerCorrectness
    RAGAS_HAS_ANSWER_CORRECTNESS = True
except Exception:
    try:
        from ragas.metrics import answer_correctness as AnswerCorrectness
        RAGAS_HAS_ANSWER_CORRECTNESS = True
    except Exception:
        RAGAS_HAS_ANSWER_CORRECTNESS = False

# -----------------------------
# 비동기 OpenAI client
# -----------------------------
async_openai_client = AsyncOpenAI(api_key=openai_key)

# -----------------------------
# ragas LLM / embeddings
# -----------------------------
ragas_llm = llm_factory(
    "gpt-4o-mini",
    client=async_openai_client
)

ragas_embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
    client=async_openai_client
)

# -----------------------------
# 기본 metric 4종
# -----------------------------
ragas_metrics = {
    "answer_relevancy": AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embedding),
    "faithfulness": Faithfulness(llm=ragas_llm),
    "context_recall": ContextRecall(llm=ragas_llm),
    "context_precision": ContextPrecision(llm=ragas_llm),
}

# -----------------------------
# 가능하면 answer_correctness 추가
# -----------------------------
if RAGAS_HAS_ANSWER_CORRECTNESS:
    try:
        ragas_metrics["answer_correctness"] = AnswerCorrectness(
            llm=ragas_llm,
            embeddings=ragas_embedding
        )
        print("answer_correctness metric: 추가됨")
    except Exception as e:
        print("answer_correctness metric 추가 실패:", e)
else:
    print("answer_correctness metric: 현재 ragas 버전에서 생략")

print("활성화된 RAGAS metric:", list(ragas_metrics.keys()))

D:\anaconda3\envs\aiffel\Lib\site-packages\instructor\providers\gemini\client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]


ragas version: 0.4.3
answer_correctness metric: 추가됨
활성화된 RAGAS metric: ['answer_relevancy', 'faithfulness', 'context_recall', 'context_precision', 'answer_correctness']


In [52]:
# ============================================================
# [CELL 22] 4단계 평가: RAGAS 평가 실행 (row-level + summary-level)
# - 목적:
#   1) 질문 단위(row-level) 점수 계산
#   2) 시스템 단위(summary-level) 평균 계산
#   3) 어떤 질문 유형에서 약한지도 분석 가능하게 저장
#
# - 품질 개선 포인트:
#   * row-level 결과를 남겨서 "왜 점수가 낮았는지" 추적 가능
#   * system 평균만 보지 않고 question_type별 집계도 가능
# ============================================================

async def score_one_row_ragas(row: pd.Series) -> Dict[str, Any]:
    """
    단일 question-answer-context row에 대해 RAGAS 점수를 계산합니다.
    """
    result = {}

    # answer_relevancy
    ar = await ragas_metrics["answer_relevancy"].ascore(
        user_input=row["question"],
        response=row["answer"]
    )
    result["answer_relevancy"] = getattr(ar, "value", ar)

    # faithfulness
    fa = await ragas_metrics["faithfulness"].ascore(
        user_input=row["question"],
        response=row["answer"],
        retrieved_contexts=row["contexts"]
    )
    result["faithfulness"] = getattr(fa, "value", fa)

    # context_recall
    cr = await ragas_metrics["context_recall"].ascore(
        user_input=row["question"],
        reference=row["ground_truth"],
        retrieved_contexts=row["contexts"]
    )
    result["context_recall"] = getattr(cr, "value", cr)

    # context_precision
    cp = await ragas_metrics["context_precision"].ascore(
        user_input=row["question"],
        reference=row["ground_truth"],
        retrieved_contexts=row["contexts"]
    )
    result["context_precision"] = getattr(cp, "value", cp)

    # 선택적 answer_correctness
    if "answer_correctness" in ragas_metrics:
        try:
            ac = await ragas_metrics["answer_correctness"].ascore(
                user_input=row["question"],
                response=row["answer"],
                reference=row["ground_truth"]
            )
            result["answer_correctness"] = getattr(ac, "value", ac)
        except Exception as e:
            result["answer_correctness"] = None
            result["answer_correctness_error"] = f"{type(e).__name__}: {str(e)}"

    return result

async def evaluate_ragas_async(df: pd.DataFrame):
    """
    전체 eval_df를 받아
    1) row-level ragas 결과
    2) system-level summary
    3) system x question_type summary
    를 반환합니다.
    """
    row_results = []

    for idx, row in df.iterrows():
        print(f"[RAGAS] {idx+1}/{len(df)} | system={row['system']} | q={row['question'][:40]}")
        metric_scores = await score_one_row_ragas(row)

        merged = row.to_dict()
        merged.update(metric_scores)
        row_results.append(merged)

    ragas_row_df = pd.DataFrame(row_results)

    metric_cols = [
        c for c in [
            "answer_relevancy",
            "faithfulness",
            "context_recall",
            "context_precision",
            "answer_correctness"
        ] if c in ragas_row_df.columns
    ]

    ragas_summary_df = (
        ragas_row_df
        .groupby("system", as_index=False)[metric_cols]
        .mean(numeric_only=True)
    )

    ragas_type_summary_df = (
        ragas_row_df
        .groupby(["system", "question_type"], as_index=False)[metric_cols]
        .mean(numeric_only=True)
    )

    return ragas_row_df, ragas_summary_df, ragas_type_summary_df

# -----------------------------
# Jupyter에서는 아래 한 줄을 실행하면 됩니다.
# -----------------------------
ragas_row_df, ragas_summary_df, ragas_type_summary_df = await evaluate_ragas_async(eval_df)

print("RAGAS row-level shape:", ragas_row_df.shape)
print("RAGAS summary shape:", ragas_summary_df.shape)
display(ragas_summary_df.sort_values(by="faithfulness", ascending=False))

[RAGAS] 1/80 | system=naive | q=이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?
[RAGAS] 2/80 | system=naive | q=주인공이 다니던 학교는 어떤 종류의 학교였나요?
[RAGAS] 3/80 | system=naive | q=이 이야기에서 '두 세계'의 주제는 무엇을 의미하나요?
[RAGAS] 4/80 | system=naive | q=프란츠 크로머는 어떤 인물인가요?
[RAGAS] 5/80 | system=naive | q=주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?
[RAGAS] 6/80 | system=naive | q=이 이야기에서 주인공의 감정은 어떤가요?
[RAGAS] 7/80 | system=naive | q=주인공이 어린 시절의 기억을 회상하는 이유는 무엇인가요?
[RAGAS] 8/80 | system=naive | q=이야기에서 '죄'의 개념은 어떻게 다루어지고 있나요?
[RAGAS] 9/80 | system=semantic | q=이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?
[RAGAS] 10/80 | system=semantic | q=주인공이 다니던 학교는 어떤 종류의 학교였나요?
[RAGAS] 11/80 | system=semantic | q=이 이야기에서 '두 세계'의 주제는 무엇을 의미하나요?
[RAGAS] 12/80 | system=semantic | q=프란츠 크로머는 어떤 인물인가요?
[RAGAS] 13/80 | system=semantic | q=주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?
[RAGAS] 14/80 | system=semantic | q=이 이야기에서 주인공의 감정은 어떤가요?
[RAGAS] 15/80 | system=semantic | q=주인공이 어린 시절의 기억을 회상하는 이유는 무엇인가요?
[RAGAS] 16/80 | system=semantic | q=이야기에서 '죄'의 개념

,system,answer_relevancy,faithfulness,context_recall,context_precision,answer_correctness
1,hybrid,0.333321,0.834375,0.750,0.480208,0.269495
0,compression,0.391628,0.833333,0.750,0.569444,0.378756
2,hyde,0.404303,0.750000,0.750,0.604167,0.290600
7,router_modular,0.326265,0.743750,0.750,0.413542,0.227241
4,multiquery,0.414285,0.739583,0.750,0.597569,0.279529
6,rerank,0.279606,0.725000,0.500,0.329861,0.162966
5,naive,0.334524,0.718750,0.625,0.625000,0.297557
8,self_rag_lite,0.345981,0.716964,0.625,0.430729,0.199457
3,iterative,0.281377,0.672917,0.875,0.396875,0.244704
9,semantic,0.260293,0.624527,0.625,0.371528,0.219354


In [53]:
# ============================================================
# [CELL 23] 4단계 평가: G-EVAL 5회 반복 평가
# - 목적:
#   1) LLM-as-a-judge 기반 정성 평가를 추가
#   2) 질문별로 5회 반복해서 평균/표준편차를 저장
#   3) RAGAS와 별도로 "답변의 실전 품질"을 보조적으로 확인
#
# - 점수 체계:
#   * 1~5점
#   * groundedness(근거충실성), relevance(질문적합성),
#     completeness(충분성), clarity(명료성), overall_score
#
#   * G-EVAL은 학습하는 것이 아니라 "반복 평가"
# ============================================================

from langchain_openai import ChatOpenAI

G_EVAL_REPEAT = 5 #시간을 보고 5~20으로 수정

judge_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=openai_key
)

def parse_first_json_object(text: str) -> Dict[str, Any]:
    """
    LLM 응답에서 첫 JSON object를 추출하는 방어적 파서
    """
    text = text.strip()

    # fenced block 제거
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    # 전체가 JSON이면 바로 파싱
    try:
        return json.loads(text)
    except Exception:
        pass

    # 첫 { ... } 블록 추출
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        return json.loads(match.group(0))

    raise ValueError("JSON object를 찾지 못했습니다.")

def build_g_eval_prompt(row: pd.Series) -> str:
    """
    G-EVAL용 judge prompt 생성
    """
    contexts_text = "\n\n".join([
        f"[context {i+1}] {c}" for i, c in enumerate(row["contexts"])
    ]) if row["contexts"] else "[context 없음]"

    prompt = f"""
당신은 RAG 응답 품질을 평가하는 엄격한 심사자입니다.
아래 기준에 따라 답변을 평가하세요.

[질문]
{row["question"]}

[기준답(reference)]
{row["ground_truth"]}

[필수 포인트]
{row.get("required_points_text", "")}

[평가 초점]
{row.get("eval_focus", "")}

[검색 문맥]
{contexts_text}

[모델 답변]
{row["answer"]}

[평가 기준]
1. groundedness: 검색 문맥에 근거해 답했는가
2. relevance: 질문에 직접 답했는가
3. completeness: 기준답과 필수 포인트를 충분히 반영했는가
4. clarity: 답변이 자연스럽고 분명한가
5. overall_score: 전체 종합 점수

[채점 방식]
- 각 항목은 1~5의 정수
- 너무 후하게 주지 말 것
- 문맥에 없는 내용을 자신있게 말하면 groundedness를 낮게 줄 것
- 기준답 핵심을 놓치면 completeness를 낮게 줄 것

[출력 형식]
반드시 아래 JSON object 하나만 출력하세요.
{{
  "groundedness": 1,
  "relevance": 1,
  "completeness": 1,
  "clarity": 1,
  "overall_score": 1,
  "reason": "한두 문장 근거"
}}
""".strip()

    return prompt

def g_eval_once(row: pd.Series) -> Dict[str, Any]:
    """
    단일 row에 대해 G-EVAL 1회 수행
    """
    prompt = build_g_eval_prompt(row)
    raw = judge_llm.invoke(prompt).content
    parsed = parse_first_json_object(raw)

    return {
        "groundedness": int(parsed["groundedness"]),
        "relevance": int(parsed["relevance"]),
        "completeness": int(parsed["completeness"]),
        "clarity": int(parsed["clarity"]),
        "overall_score": int(parsed["overall_score"]),
        "reason": normalize_whitespace(parsed.get("reason", "")),
    }

def g_eval_repeat(row: pd.Series, n_repeat: int = 5) -> Dict[str, Any]:
    """
    단일 row에 대해 G-EVAL을 여러 번 반복하고 평균/표준편차를 계산
    """
    runs = []
    for r in range(n_repeat):
        try:
            result = g_eval_once(row)
            result["repeat_idx"] = r + 1
            runs.append(result)
        except Exception as e:
            runs.append({
                "groundedness": None,
                "relevance": None,
                "completeness": None,
                "clarity": None,
                "overall_score": None,
                "reason": f"[ERROR] {type(e).__name__}: {str(e)}",
                "repeat_idx": r + 1,
            })

    valid_runs = [x for x in runs if x["overall_score"] is not None]

    def avg_of(key: str):
        vals = [x[key] for x in valid_runs if x.get(key) is not None]
        return sum(vals) / len(vals) if vals else None

    def std_of(key: str):
        vals = [x[key] for x in valid_runs if x.get(key) is not None]
        return statistics.pstdev(vals) if len(vals) >= 2 else 0.0 if len(vals) == 1 else None

    return {
        "g_eval_groundedness_mean": avg_of("groundedness"),
        "g_eval_relevance_mean": avg_of("relevance"),
        "g_eval_completeness_mean": avg_of("completeness"),
        "g_eval_clarity_mean": avg_of("clarity"),
        "g_eval_overall_mean": avg_of("overall_score"),
        "g_eval_overall_std": std_of("overall_score"),
        "g_eval_valid_runs": len(valid_runs),
        "g_eval_reasons": [x["reason"] for x in runs],
        "g_eval_raw_runs": runs,
    }

g_eval_row_results = []

for idx, row in ragas_row_df.iterrows():
    print(f"[G-EVAL] {idx+1}/{len(ragas_row_df)} | system={row['system']} | q={row['question'][:40]}")
    g_result = g_eval_repeat(row, n_repeat=G_EVAL_REPEAT)

    merged = row.to_dict()
    merged.update(g_result)
    g_eval_row_results.append(merged)

g_eval_row_df = pd.DataFrame(g_eval_row_results)

g_eval_summary_df = (
    g_eval_row_df
    .groupby("system", as_index=False)[[
        "g_eval_groundedness_mean",
        "g_eval_relevance_mean",
        "g_eval_completeness_mean",
        "g_eval_clarity_mean",
        "g_eval_overall_mean",
        "g_eval_overall_std",
    ]]
    .mean(numeric_only=True)
)

g_eval_type_summary_df = (
    g_eval_row_df
    .groupby(["system", "question_type"], as_index=False)[[
        "g_eval_groundedness_mean",
        "g_eval_relevance_mean",
        "g_eval_completeness_mean",
        "g_eval_clarity_mean",
        "g_eval_overall_mean",
        "g_eval_overall_std",
    ]]
    .mean(numeric_only=True)
)

print("G-EVAL row-level shape:", g_eval_row_df.shape)
print("G-EVAL summary shape:", g_eval_summary_df.shape)
display(g_eval_summary_df.sort_values(by="g_eval_overall_mean", ascending=False))

[G-EVAL] 1/80 | system=naive | q=이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?
[G-EVAL] 2/80 | system=naive | q=주인공이 다니던 학교는 어떤 종류의 학교였나요?
[G-EVAL] 3/80 | system=naive | q=이 이야기에서 '두 세계'의 주제는 무엇을 의미하나요?
[G-EVAL] 4/80 | system=naive | q=프란츠 크로머는 어떤 인물인가요?
[G-EVAL] 5/80 | system=naive | q=주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?
[G-EVAL] 6/80 | system=naive | q=이 이야기에서 주인공의 감정은 어떤가요?
[G-EVAL] 7/80 | system=naive | q=주인공이 어린 시절의 기억을 회상하는 이유는 무엇인가요?
[G-EVAL] 8/80 | system=naive | q=이야기에서 '죄'의 개념은 어떻게 다루어지고 있나요?
[G-EVAL] 9/80 | system=semantic | q=이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?
[G-EVAL] 10/80 | system=semantic | q=주인공이 다니던 학교는 어떤 종류의 학교였나요?
[G-EVAL] 11/80 | system=semantic | q=이 이야기에서 '두 세계'의 주제는 무엇을 의미하나요?
[G-EVAL] 12/80 | system=semantic | q=프란츠 크로머는 어떤 인물인가요?
[G-EVAL] 13/80 | system=semantic | q=주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?
[G-EVAL] 14/80 | system=semantic | q=이 이야기에서 주인공의 감정은 어떤가요?
[G-EVAL] 15/80 | system=semantic | q=주인공이 어린 시절의 기억을 회상하는 이유는 무엇인가요?
[G-EVAL] 16/80 | system=semantic |

,system,g_eval_groundedness_mean,g_eval_relevance_mean,g_eval_completeness_mean,g_eval_clarity_mean,g_eval_overall_mean,g_eval_overall_std
0,compression,4.375,4.500,4.250,4.375,4.375,0.000000
1,hybrid,3.800,4.375,3.675,4.000,3.775,0.050000
4,multiquery,3.650,4.275,3.525,3.875,3.650,0.050000
2,hyde,3.625,4.125,3.400,3.875,3.625,0.000000
5,naive,3.625,4.000,3.625,3.750,3.625,0.000000
3,iterative,3.475,4.000,3.475,3.475,3.475,0.050000
7,router_modular,3.450,4.200,3.425,3.650,3.450,0.111237
8,self_rag_lite,3.400,3.900,3.375,3.650,3.400,0.050000
6,rerank,3.150,3.375,3.150,3.275,3.150,0.050000
9,semantic,3.150,3.500,3.150,3.150,3.150,0.050000


In [54]:
# ============================================================
# [CELL 24] 4단계 평가: RAGAS + G-EVAL 통합 결과 정리 / 저장
# - 목적:
#   1) RAGAS와 G-EVAL을 한 표로 합쳐 비교
#   2) row-level / system-level / type-level 결과를 모두 저장
#   3) faithfulness + g_eval_overall_mean을 함께 보며 해석 가능하게 함
# ============================================================

from IPython.display import display

# -----------------------------
# 1) 시스템 단위 통합 summary
# -----------------------------
final_summary_df = ragas_summary_df.merge(
    g_eval_summary_df,
    on="system",
    how="left"
)

# 보기 좋게 반올림
final_summary_view = final_summary_df.copy()
for c in final_summary_view.columns:
    if c != "system":
        final_summary_view[c] = final_summary_view[c].round(4)

print("=== RAGAS + G-EVAL 통합 비교표 ===")
display(
    final_summary_view.sort_values(
        by=["faithfulness", "g_eval_overall_mean"],
        ascending=[False, False]
    )
)

# -----------------------------
# 2) 질문 유형(question_type) 기준 통합 summary
# -----------------------------
final_type_summary_df = ragas_type_summary_df.merge(
    g_eval_type_summary_df,
    on=["system", "question_type"],
    how="left"
)

final_type_summary_view = final_type_summary_df.copy()
for c in final_type_summary_view.columns:
    if c not in ["system", "question_type"]:
        final_type_summary_view[c] = final_type_summary_view[c].round(4)

print("=== question_type 기준 비교표 ===")
display(
    final_type_summary_view.sort_values(
        by=["question_type", "faithfulness", "g_eval_overall_mean"],
        ascending=[True, False, False]
    )
)

# -----------------------------
# 3) 저장
# -----------------------------
eval_samples_df = preview_eval_samples(eval_samples)

eval_samples_df.to_csv("./eval_samples_high_quality.csv", index=False, encoding="utf-8-sig")
eval_df.to_csv("./eval_predictions_raw.csv", index=False, encoding="utf-8-sig")
ragas_row_df.to_csv("./ragas_row_result.csv", index=False, encoding="utf-8-sig")
ragas_summary_df.to_csv("./ragas_summary_result.csv", index=False, encoding="utf-8-sig")
g_eval_row_df.to_csv("./g_eval_row_result.csv", index=False, encoding="utf-8-sig")
g_eval_summary_df.to_csv("./g_eval_summary_result.csv", index=False, encoding="utf-8-sig")
final_summary_df.to_csv("./ragas_g_eval_final_summary.csv", index=False, encoding="utf-8-sig")
final_type_summary_df.to_csv("./ragas_g_eval_type_summary.csv", index=False, encoding="utf-8-sig")

print("저장 완료:")
print("- ./eval_samples_high_quality.csv")
print("- ./eval_predictions_raw.csv")
print("- ./ragas_row_result.csv")
print("- ./ragas_summary_result.csv")
print("- ./g_eval_row_result.csv")
print("- ./g_eval_summary_result.csv")
print("- ./ragas_g_eval_final_summary.csv")
print("- ./ragas_g_eval_type_summary.csv")

=== RAGAS + G-EVAL 통합 비교표 ===


,system,answer_relevancy,faithfulness,context_recall,context_precision,answer_correctness,g_eval_groundedness_mean,g_eval_relevance_mean,g_eval_completeness_mean,g_eval_clarity_mean,g_eval_overall_mean,g_eval_overall_std
1,hybrid,0.3333,0.8344,0.750,0.4802,0.2695,3.800,4.375,3.675,4.000,3.775,0.0500
0,compression,0.3916,0.8333,0.750,0.5694,0.3788,4.375,4.500,4.250,4.375,4.375,0.0000
2,hyde,0.4043,0.7500,0.750,0.6042,0.2906,3.625,4.125,3.400,3.875,3.625,0.0000
7,router_modular,0.3263,0.7438,0.750,0.4135,0.2272,3.450,4.200,3.425,3.650,3.450,0.1112
4,multiquery,0.4143,0.7396,0.750,0.5976,0.2795,3.650,4.275,3.525,3.875,3.650,0.0500
6,rerank,0.2796,0.7250,0.500,0.3299,0.1630,3.150,3.375,3.150,3.275,3.150,0.0500
5,naive,0.3345,0.7188,0.625,0.6250,0.2976,3.625,4.000,3.625,3.750,3.625,0.0000
8,self_rag_lite,0.3460,0.7170,0.625,0.4307,0.1995,3.400,3.900,3.375,3.650,3.400,0.0500
3,iterative,0.2814,0.6729,0.875,0.3969,0.2447,3.475,4.000,3.475,3.475,3.475,0.0500
9,semantic,0.2603,0.6245,0.625,0.3715,0.2194,3.150,3.500,3.150,3.150,3.150,0.0500


=== question_type 기준 비교표 ===


,system,question_type,answer_relevancy,faithfulness,context_recall,context_precision,answer_correctness,g_eval_groundedness_mean,g_eval_relevance_mean,g_eval_completeness_mean,g_eval_clarity_mean,g_eval_overall_mean,g_eval_overall_std
0,compression,character_or_entity,0.4774,1.0000,1.0,1.0000,0.3457,5.0,5.0,5.0,5.0,5.0,0.0000
12,hyde,character_or_entity,0.7813,1.0000,1.0,0.5000,0.5346,5.0,5.0,5.0,5.0,5.0,0.0000
24,multiquery,character_or_entity,0.7696,1.0000,1.0,1.0000,0.5468,5.0,5.0,5.0,5.0,5.0,0.0000
54,semantic,character_or_entity,0.2114,1.0000,1.0,0.2500,0.3944,5.0,5.0,5.0,5.0,5.0,0.0000
30,naive,character_or_entity,0.0000,1.0000,0.0,0.0000,0.1103,1.0,1.0,1.0,1.0,1.0,0.0000
36,rerank,character_or_entity,0.0000,1.0000,0.0,0.0000,0.1102,1.0,1.0,1.0,1.0,1.0,0.0000
6,hybrid,character_or_entity,0.0000,0.8000,1.0,0.2500,0.5455,4.0,5.0,4.0,4.0,4.0,0.0000
18,iterative,character_or_entity,0.0000,0.8000,1.0,0.2500,0.5460,4.0,5.0,4.0,4.0,4.0,0.0000
42,router_modular,character_or_entity,0.0000,0.8000,1.0,0.2500,0.3875,4.0,5.0,4.0,4.0,4.0,0.0000
48,self_rag_lite,character_or_entity,0.0000,0.5714,0.0,0.0000,0.1046,2.0,2.0,2.0,3.0,2.0,0.0000


저장 완료:
- ./eval_samples_high_quality.csv
- ./eval_predictions_raw.csv
- ./ragas_row_result.csv
- ./ragas_summary_result.csv
- ./g_eval_row_result.csv
- ./g_eval_summary_result.csv
- ./ragas_g_eval_final_summary.csv
- ./ragas_g_eval_type_summary.csv


In [55]:
# ============================================================
# [CELL 25] 4단계 평가: 실패/약점 분석용 뷰
# - 목적:
#   1) 점수가 낮은 질문을 바로 확인
#   2) 어떤 시스템이 어떤 질문유형에서 약한지 빠르게 확인
#   3) 이후 retrieval 개선 방향을 정하기 쉽게 함
# ============================================================

# -----------------------------
# 1) RAGAS 기준 하위 케이스
# -----------------------------
ragas_failure_view = g_eval_row_df.copy()

sort_candidates = [
    c for c in [
        "faithfulness",
        "context_precision",
        "context_recall",
        "answer_relevancy",
        "g_eval_overall_mean"
    ] if c in ragas_failure_view.columns
]

print("=== 하위 케이스 확인용 표 ===")
display_cols = [
    "system",
    "question_type",
    "question",
    "reference",
    "required_points_text",
    "answer",
    "num_contexts",
] + sort_candidates

display(
    ragas_failure_view[display_cols]
    .sort_values(
        by=["faithfulness", "g_eval_overall_mean"],
        ascending=[True, True]
    )
    .head(15)
)

# -----------------------------
# 2) 시스템별 간단 해석 규칙
# -----------------------------
def interpret_rag_result(row: pd.Series) -> str:
    msgs = []

    if row.get("faithfulness", 1) < 0.85:
        msgs.append("faithfulness 낮음 → 답변 생성 단계 hallucination 가능성")
    if row.get("context_recall", 1) < 0.80:
        msgs.append("context_recall 낮음 → retrieval 누락 가능성")
    if row.get("context_precision", 1) < 0.50:
        msgs.append("context_precision 낮음 → 불필요 문맥 과다, rerank/compression 재검토 필요")
    if row.get("answer_relevancy", 1) < 0.80:
        msgs.append("answer_relevancy 낮음 → 질문 의도 해석 또는 답변 집중도 부족")
    if row.get("g_eval_overall_mean", 5) < 3.5:
        msgs.append("G-EVAL 낮음 → 실사용 답변 품질이 체감상 부족")

    if not msgs:
        msgs.append("전반적으로 안정적")

    return " | ".join(msgs)

final_interpret_df = final_summary_df.copy()
final_interpret_df["interpretation"] = final_interpret_df.apply(interpret_rag_result, axis=1)

print("=== 시스템별 해석 ===")
display(
    final_interpret_df[[
        "system",
        "answer_relevancy",
        "faithfulness",
        "context_recall",
        "context_precision",
        "g_eval_overall_mean",
        "interpretation"
    ]].sort_values(by=["faithfulness", "g_eval_overall_mean"], ascending=[False, False])
)

=== 하위 케이스 확인용 표 ===


,system,question_type,question,reference,required_points_text,answer,num_contexts,faithfulness,context_precision,context_recall,answer_relevancy,g_eval_overall_mean
4,naive,reasoning_light,주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?,주인공은 프란츠가 자신을 고발할까 두려워 시계를 주기로 결심했다.,고발에 대한 두려움 / 시계의 가치,문서에서 주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 확인되지 않습니다.,4,0.000000,0.000000,0.0,0.000000,1.0
14,semantic,summary,주인공이 어린 시절의 기억을 회상하는 이유는 무엇인가요?,주인공은 자신의 과거를 통해 현재의 자신을 이해하고자 한다.,과거 회상 / 현재 이해,문서에서 주인공이 어린 시절의 기억을 회상하는 이유는 확인되지 않습니다.,4,0.000000,0.000000,0.0,0.000000,1.0
20,multiquery,reasoning_light,주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?,주인공은 프란츠가 자신을 고발할까 두려워 시계를 주기로 결심했다.,고발에 대한 두려움 / 시계의 가치,문서에서 주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 확인되지 않습니다.,6,0.000000,0.000000,0.0,0.000000,1.0
28,rerank,reasoning_light,주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?,주인공은 프란츠가 자신을 고발할까 두려워 시계를 주기로 결심했다.,고발에 대한 두려움 / 시계의 가치,문서에서 주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 확인되지 않습니다.,4,0.000000,0.000000,0.0,0.000000,1.0
36,compression,reasoning_light,주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?,주인공은 프란츠가 자신을 고발할까 두려워 시계를 주기로 결심했다.,고발에 대한 두려움 / 시계의 가치,문서에서 주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 확인되지 않습니다.,4,0.000000,0.000000,0.0,0.000000,1.0
44,hyde,reasoning_light,주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?,주인공은 프란츠가 자신을 고발할까 두려워 시계를 주기로 결심했다.,고발에 대한 두려움 / 시계의 가치,문서에서 주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 확인되지 않습니다.,4,0.000000,0.000000,0.0,0.000000,1.0
52,hybrid,reasoning_light,주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?,주인공은 프란츠가 자신을 고발할까 두려워 시계를 주기로 결심했다.,고발에 대한 두려움 / 시계의 가치,문서에서 주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 확인되지 않습니다.,6,0.000000,0.000000,0.0,0.000000,1.0
60,self_rag_lite,reasoning_light,주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?,주인공은 프란츠가 자신을 고발할까 두려워 시계를 주기로 결심했다.,고발에 대한 두려움 / 시계의 가치,문서에서 주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 확인되지 않습니다.,6,0.000000,0.000000,0.0,0.000000,1.0
65,iterative,factual,주인공이 다니던 학교는 어떤 종류의 학교였나요?,주인공은 작은 시골 마을의 문법 학교에 다녔다.,작은 시골 마을 / 문법 학교,주인공이 다니던 학교에 대한 구체적인 정보는 문서에서 확인되지 않습니다.,6,0.000000,0.000000,0.0,0.000000,1.0
68,iterative,reasoning_light,주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 무엇인가요?,주인공은 프란츠가 자신을 고발할까 두려워 시계를 주기로 결심했다.,고발에 대한 두려움 / 시계의 가치,문서에서 주인공이 프란츠에게 자신의 시계를 주기로 결심한 이유는 확인되지 않습니다.,6,0.000000,0.000000,1.0,0.000000,1.0


=== 시스템별 해석 ===


,system,answer_relevancy,faithfulness,context_recall,context_precision,g_eval_overall_mean,interpretation
1,hybrid,0.333321,0.834375,0.750,0.480208,3.775,faithfulness 낮음 → 답변 생성 단계 hallucination 가능성 |...
0,compression,0.391628,0.833333,0.750,0.569444,4.375,faithfulness 낮음 → 답변 생성 단계 hallucination 가능성 |...
2,hyde,0.404303,0.750000,0.750,0.604167,3.625,faithfulness 낮음 → 답변 생성 단계 hallucination 가능성 |...
7,router_modular,0.326265,0.743750,0.750,0.413542,3.450,faithfulness 낮음 → 답변 생성 단계 hallucination 가능성 |...
4,multiquery,0.414285,0.739583,0.750,0.597569,3.650,faithfulness 낮음 → 답변 생성 단계 hallucination 가능성 |...
6,rerank,0.279606,0.725000,0.500,0.329861,3.150,faithfulness 낮음 → 답변 생성 단계 hallucination 가능성 |...
5,naive,0.334524,0.718750,0.625,0.625000,3.625,faithfulness 낮음 → 답변 생성 단계 hallucination 가능성 |...
8,self_rag_lite,0.345981,0.716964,0.625,0.430729,3.400,faithfulness 낮음 → 답변 생성 단계 hallucination 가능성 |...
3,iterative,0.281377,0.672917,0.875,0.396875,3.475,faithfulness 낮음 → 답변 생성 단계 hallucination 가능성 |...
9,semantic,0.260293,0.624527,0.625,0.371528,3.150,faithfulness 낮음 → 답변 생성 단계 hallucination 가능성 |...


In [56]:
# ============================================================
# [CELL 26] 4단계 평가: 선택사항 - 같은 질문에 대한 시스템별 답변 직접 비교
# - 목적:
#   1) 점수표만 보지 말고 실제 답변을 같이 확인
#   2) 특히 faithfulness나 g_eval이 낮은 시스템은 실제 문장을 확인해야 함
# ============================================================

compare_question = eval_samples[0]["question"] if len(eval_samples) > 0 else "이 문서의 핵심 주제는 무엇인가요?"

compare_rows = []

for name in systems_to_compare:
    try:
        result = system_registry[name](compare_question)

        answer_text = result.get("answer", "")
        contexts = clean_contexts(result.get("contexts", []))

        compare_rows.append({
            "system": name,
            "question": compare_question,
            "answer": answer_text[:1500],
            "num_contexts": len(contexts),
            "contexts_preview": " || ".join([c[:200] for c in contexts[:2]])
        })

    except Exception as e:
        compare_rows.append({
            "system": name,
            "question": compare_question,
            "answer": f"[ERROR] {type(e).__name__}: {str(e)}",
            "num_contexts": None,
            "contexts_preview": None
        })

compare_df = pd.DataFrame(compare_rows)
display(compare_df)

,system,question,answer,num_contexts,contexts_preview
0,naive,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,주인공이 자신의 이야기를 시작하는 이유는 자신의 이야기를 전하기 위해서는 어린 시절...,4,Prologue I cannot tell my story without going ...
1,semantic,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,주인공이 자신의 이야기를 시작하는 이유는 그가 내면의 갈등과 진정한 자아를 찾기 위...,4,""" he said laughing. ""You must always ask; you ..."
2,multiquery,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,주인공이 자신의 이야기를 시작하는 이유는 자신의 내면의 갈등과 성장 과정을 공유하고...,6,""" he said laughing. ""You must always ask; you ..."
3,rerank,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,"주인공이 자신의 이야기를 시작하는 이유는 세상이 전쟁을 필요로 하고, 그로 인해 자...",4,I must go to you. I had just been telling her ...
4,compression,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,주인공이 자신의 이야기를 시작하는 이유는 자신의 이야기가 중요하고 개인적이기 때문입...,4,In this connexion I remember having heard that...
5,hyde,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,주인공이 자신의 이야기를 시작하는 이유는 자신이 겪고 있는 내적 갈등과 고뇌를 극복...,4,I was at that time an unusual youth .for my ei...
6,hybrid,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,"주인공이 자신의 이야기를 시작하는 이유는 그가 내면의 목소리를 듣고, 자신의 정체성...",6,""" he said laughing. ""You must always ask; you ..."
7,self_rag_lite,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,주인공이 자신의 이야기를 시작하는 이유는 내면의 갈등과 자아 발견의 과정을 통해 자...,6,""" he said laughing. ""You must always ask; you ..."
8,iterative,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,"주인공이 자신의 이야기를 시작하는 이유는 그가 내면의 목소리를 듣고, 자신의 정체성...",6,""" he said laughing. ""You must always ask; you ..."
9,router_modular,이 이야기의 주인공이 자신의 이야기를 시작하는 이유는 무엇인가요?,"주인공이 자신의 이야기를 시작하는 이유는 ""이제 이야기를 마무리 지으려 한다""는 언...",6,""" he said laughing. ""You must always ask; you ..."
